In [2]:
"""
Module: Session Anomaly Detection
Description: This script loads a downsized network flow dataset, performs feature engineering,
and trains a lightweight Feed-Forward Neural Network (FFNN) for Hijack/Bot detection.
Crucially, it exports the trained model to the ONNX format.
"""

import os
# ------Forced CPU execution--------
# IBM Z's Telum processor handles AI inferencing directly on-chip.
# Disabling CUDA prevents environment initialization errors on s390x architecture
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import tf2onnx

DATA_PATH = '../data/cic_ids_downsized.csv'
MODEL_OUTPUT_PATH = '../models/session_anomaly.onnx'

def prepare_data(file_path: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Loads raw session telemetry data and applies feature engineering.
    Optimized for processing network data commonly found in systems like IBM Z.
    """
    # Fallback to generate synthetic data if the CSV isn't mounted yet for the datathon
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found. Generating synthetic CIC-IDS data for export.")
        X = np.random.rand(1000, 3).astype(np.float32)
        y = np.random.randint(0, 2, 1000).astype(np.float32)
        return X, y

    df = pd.read_csv(file_path)

    # Logarithmic transformation flattens massive variance in byte transfer rates
    df['log_bytes_per_sec'] = np.log1p(df['flow_bytes_per_sec'])

    features = [
        'flow_duration',
        'log_bytes_per_sec',
        'packet_rate'
    ]

    # Return as float32 tensors to ensure strict compatibility with ONNX
    return df[features].values.astype(np.float32), df['is_anomaly'].values.astype(np.float32)

def build_model(input_dim: int) -> Sequential:
    """
    Constructs a lightweight Feed-Forward Neural Network architecture.
    Designed specifically for millisecond-level inference speeds.
    """
    model = Sequential([
        tf.keras.Input(shape=(input_dim,), name="telemetry_features"),
        Dense(16, activation='relu'),
        Dropout(0.2),
        Dense(8, activation='relu'),
        # Sigmoid activation cleanly compresses the final output into a 0.0 to 1.0 probability scalar.
        Dense(1, activation='sigmoid', name="anomaly_probability")
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def export_to_onnx(keras_model: Sequential, output_path: str, input_dim: int) -> None:
    """
    Safely exports the model to the environment-agnostic ONNX format supported by IBM Z.
    We use direct graph mapping (tf.function) to bypass known tf2onnx compatibility issues.
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    @tf.function
    def model_fn(x):
        # CRITICAL FOR PRODUCTION: setting training=False locks the model into deterministic prediction mode
        return keras_model(x, training=False)

    spec = [tf.TensorSpec((None, input_dim), tf.float32, name="telemetry_features")]

    tf2onnx.convert.from_function(
        model_fn,
        input_signature=spec,
        output_path=output_path,
        opset=13
    )

if __name__ == "__main__":
    X, y = prepare_data(DATA_PATH)
    model = build_model(input_dim=X.shape[1])
    model.fit(X, y, epochs=5, batch_size=256, validation_split=0.2)
    export_to_onnx(model, MODEL_OUTPUT_PATH, input_dim=X.shape[1])
    print(f"SUCCESS: Production session model successfully exported to {MODEL_OUTPUT_PATH}")

KeyError: 'flow_bytes_per_sec'